In [1]:
import pandas as pd
import datetime as dt

# Load cleaned data
df = pd.read_csv("cleaned_superstore.csv")

# Convert date
df['Order Date'] = pd.to_datetime(df['Order Date'])

# Set today's date (latest date in dataset)
today = df['Order Date'].max()

# Create RFM table
rfm = df.groupby('Customer ID').agg({
    'Order Date': lambda x: (today - x.max()).days,  # Recency
    'Order ID': 'count',                            # Frequency
    'Sales': 'sum'                                  # Monetary
})

# Rename columns
rfm.columns = ['Recency', 'Frequency', 'Monetary']

# Show result
rfm.head()

,Recency,Frequency,Monetary
Customer ID,,,
AA-10315,184,11,5563.560
AA-10375,19,15,1056.390
AA-10480,259,12,1790.512
AA-10645,55,18,5086.935
AB-10015,415,6,886.156


In [2]:
rfm['R_score'] = pd.qcut(rfm['Recency'], 4, labels=[4,3,2,1])
rfm['F_score'] = pd.qcut(rfm['Frequency'], 4, labels=[1,2,3,4])
rfm['M_score'] = pd.qcut(rfm['Monetary'], 4, labels=[1,2,3,4])

rfm.head()

,Recency,Frequency,Monetary,R_score,F_score,M_score
Customer ID,,,,,,
AA-10315,184,11,5563.560,2,2,4
AA-10375,19,15,1056.390,4,3,1
AA-10480,259,12,1790.512,1,3,2
AA-10645,55,18,5086.935,3,4,4
AB-10015,415,6,886.156,1,1,1


In [3]:
rfm['RFM_Score'] = rfm['R_score'].astype(str) + rfm['F_score'].astype(str) + rfm['M_score'].astype(str)

rfm.head()

,Recency,Frequency,Monetary,R_score,F_score,M_score,RFM_Score
Customer ID,,,,,,,
AA-10315,184,11,5563.560,2,2,4,224
AA-10375,19,15,1056.390,4,3,1,431
AA-10480,259,12,1790.512,1,3,2,132
AA-10645,55,18,5086.935,3,4,4,344
AB-10015,415,6,886.156,1,1,1,111


In [4]:
def segment(row):
    if row['RFM_Score'] == '444':
        return 'Best Customers'
    elif row['F_score'] >= 3:
        return 'Loyal Customers'
    else:
        return 'Others'

rfm['Segment'] = rfm.apply(segment, axis=1)

rfm.head()

,Recency,Frequency,Monetary,R_score,F_score,M_score,RFM_Score,Segment
Customer ID,,,,,,,,
AA-10315,184,11,5563.560,2,2,4,224,Others
AA-10375,19,15,1056.390,4,3,1,431,Loyal Customers
AA-10480,259,12,1790.512,1,3,2,132,Loyal Customers
AA-10645,55,18,5086.935,3,4,4,344,Loyal Customers
AB-10015,415,6,886.156,1,1,1,111,Others


In [5]:
rfm['Segment'].value_counts()

Segment
Others             399
Loyal Customers    365
Best Customers      29
Name: count, dtype: int64

In [6]:
rfm.to_csv("rfm_output.csv")